In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install -q rasterio timm albumentations
!pip install -q scikit-learn matplotlib pandas numpy

print("✅ All libraries installed!")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
import warnings
import time
warnings.filterwarnings('ignore')

try:
    import rasterio
except ImportError:
    os.system('pip install rasterio')
    import rasterio

# ===== PATHS =====
BASE_PATH   = '/kaggle/input/beyond-visible-spectrum-ai-for-agriculture-2026p2'
TRAIN_DIR   = os.path.join(BASE_PATH, 'ICPR02', 'kaggle')
EVAL_DIR    = os.path.join(BASE_PATH, 'ICPR02', 'kaggle', 'evaluation')
ARCHIVE_DIR = os.path.join(BASE_PATH, 'archive', 'share')
CLASS_NAMES = ['Aphid', 'Blast', 'RPH', 'Rust']

print("📂 Checking all paths:")
for name, path in [('TRAIN_DIR', TRAIN_DIR), ('EVAL_DIR', EVAL_DIR), ('ARCHIVE_DIR', ARCHIVE_DIR)]:
    exists = os.path.exists(path)
    count = len(os.listdir(path)) if exists else 0
    print(f"  {'✅' if exists else '❌'} {name}: {path} ({count} items)")

print(f"\n📁 Training class folders:")
if os.path.exists(TRAIN_DIR):
    for item in sorted(os.listdir(TRAIN_DIR)):
        full = os.path.join(TRAIN_DIR, item)
        if os.path.isdir(full):
            print(f"  📁 {item}/ → {len(os.listdir(full))} samples")

In [ ]:
def print_tree(path, prefix="", max_depth=3, depth=0, max_items=8):
    if depth >= max_depth: return
    try: items = sorted(os.listdir(path))
    except: return
    for i, item in enumerate(items[:max_items]):
        full = os.path.join(path, item)
        last = (i == min(len(items), max_items) - 1)
        conn = "└── " if last else "├── "
        if os.path.isdir(full):
            print(f"{prefix}{conn}📁 {item}/ ({len(os.listdir(full))} items)")
            print_tree(full, prefix + ("    " if last else "│   "), max_depth, depth+1, max_items)
        else:
            print(f"{prefix}{conn}📄 {item} ({os.path.getsize(full):,} bytes)")
    if len(items) > max_items:
        print(f"{prefix}└── ... +{len(items)-max_items} more")

print("=" * 60)
print("📂 FULL DATA STRUCTURE")
print("=" * 60)
print_tree(BASE_PATH, max_depth=4)

In [ ]:
BAND_NAMES = ['B1','B2','B3','B4','B5','B6','B7','B8','B8A','B9','B11','B12']
BAND_INFO = {
    'B1':('Coastal/Aerosol','60m','#9400D3'), 'B2':('Blue','10m','#0000FF'),
    'B3':('Green','10m','#00FF00'), 'B4':('Red','10m','#FF0000'),
    'B5':('Red Edge 1','20m','#FF4500'), 'B6':('Red Edge 2','20m','#FF6347'),
    'B7':('Red Edge 3','20m','#FF7F50'), 'B8':('NIR','10m','#8B0000'),
    'B8A':('NIR Narrow','20m','#A52A2A'), 'B9':('Water Vapour','60m','#00CED1'),
    'B11':('SWIR 1','20m','#FFD700'), 'B12':('SWIR 2','20m','#FFA500'),
}

def load_bands(folder_path):
    bands = {}
    for bn in BAND_NAMES:
        tif = os.path.join(folder_path, f"{bn}.tif")
        if os.path.exists(tif):
            with rasterio.open(tif) as src:
                bands[bn] = src.read(1).astype(np.float32)
    return bands

def normalize_band(band, pct=(2,98)):
    valid = band[band > 0]
    if len(valid) == 0: return np.zeros_like(band, dtype=np.float32)
    lo, hi = np.percentile(valid, pct)
    if hi == lo: return np.zeros_like(band, dtype=np.float32)
    return np.clip((band - lo) / (hi - lo), 0, 1)

def load_any_image(path):
    """Load image from path (folder with .tif bands, single .tif, or image file)."""
    if os.path.isdir(path):
        bands = load_bands(path)
        if all(b in bands for b in ['B4','B3','B2']):
            return np.stack([normalize_band(bands['B4']),
                           normalize_band(bands['B3']),
                           normalize_band(bands['B2'])], axis=-1), 'rgb', bands
        elif bands:
            first = list(bands.values())[0]
            return normalize_band(first), 'gray', bands
    else:
        fpath = path if os.path.isfile(path) else None
        if fpath and fpath.endswith(('.tif','.tiff')):
            with rasterio.open(fpath) as src:
                data = src.read().astype(np.float32)
                if data.shape[0] >= 3:
                    return np.stack([normalize_band(data[2]),
                                   normalize_band(data[1]),
                                   normalize_band(data[0])], axis=-1), 'rgb', None
                else:
                    return normalize_band(data[0]), 'gray', None
        elif fpath and fpath.endswith(('.png','.jpg','.jpeg')):
            from PIL import Image
            return np.array(Image.open(fpath)), 'rgb', None
    return None, None, None

# --- Class Distribution ---
print("🏷️ LABELED DATASET")
print("=" * 60)
class_counts = {}
for cls in CLASS_NAMES:
    d = os.path.join(TRAIN_DIR, cls)
    class_counts[cls] = len(os.listdir(d)) if os.path.exists(d) else 0

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']
names, counts = list(class_counts.keys()), list(class_counts.values())

bars = axes[0].bar(names, counts, color=colors, edgecolor='black', lw=0.5)
axes[0].set_title('Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Samples')
for b, c in zip(bars, counts):
    axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+1, str(c), ha='center', fontweight='bold')

axes[1].pie(counts, labels=names, autopct='%1.1f%%', colors=colors, startangle=90, explode=[0.05]*4)
axes[1].set_title('Class Distribution (%)', fontsize=13, fontweight='bold')
plt.suptitle('� Crop Disease Dataset', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.savefig('/kaggle/working/class_dist.png', dpi=150, bbox_inches='tight'); plt.show()

# --- Sample Images Per Class ---
n_show = 4
fig, axes = plt.subplots(len(CLASS_NAMES), n_show, figsize=(4*n_show, 4*len(CLASS_NAMES)))

for row, cls in enumerate(CLASS_NAMES):
    cls_dir = os.path.join(TRAIN_DIR, cls)
    samples = sorted(os.listdir(cls_dir))[:n_show] if os.path.exists(cls_dir) else []
    axes[row, 0].set_ylabel(cls, fontsize=14, fontweight='bold', rotation=0, labelpad=60)

    for col in range(n_show):
        ax = axes[row, col]
        if col < len(samples):
            img, mode, _ = load_any_image(os.path.join(cls_dir, samples[col]))
            if img is not None:
                ax.imshow(img, cmap='gray' if mode == 'gray' else None)
            ax.set_title(samples[col], fontsize=7)
        ax.axis('off')

plt.suptitle('🏷️ Sample Images by Disease Class', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.savefig('/kaggle/working/labeled_samples.png', dpi=150, bbox_inches='tight'); plt.show()

# --- Band Details for One Sample ---
print("\n📊 Inspecting one sample in detail:")
for cls in CLASS_NAMES:
    cls_dir = os.path.join(TRAIN_DIR, cls)
    if os.path.exists(cls_dir):
        sample = sorted(os.listdir(cls_dir))[0]
        sp = os.path.join(cls_dir, sample)
        img, mode, bands = load_any_image(sp)
        print(f"\n  Class: {cls} | Sample: {sample}")
        if os.path.isdir(sp):
            print(f"  Type: Folder with files: {os.listdir(sp)[:5]}")
        else:
            print(f"  Type: File ({os.path.getsize(sp):,} bytes)")
        if bands:
            for bn, bd in bands.items():
                info = BAND_INFO.get(bn, ('?','?','#000'))
                print(f"    {bn:5s} {info[0]:18s} {info[1]:5s} shape={bd.shape} "
                      f"min={bd.min():.1f} max={bd.max():.1f} mean={bd.mean():.1f}")
        break

# --- Evaluation Data Peek ---
print("\n📋 EVALUATION (Test) DATA:")
if os.path.exists(EVAL_DIR):
    items = sorted(os.listdir(EVAL_DIR))
    print(f"  {len(items)} items")
    print(f"  First 5: {items[:5]}")
    first = os.path.join(EVAL_DIR, items[0])
    if os.path.isdir(first):
        print(f"  Inside '{items[0]}': {os.listdir(first)[:5]}")
else:
    print("  ❌ Not found")

print("\n✅ Data exploration complete!")


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score, f1_score
import timm
import rasterio
import numpy as np
import pandas as pd
import os, time

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ===== PATHS =====
BASE_PATH   = '/kaggle/input/beyond-visible-spectrum-ai-for-agriculture-2026p2'
TRAIN_DIR   = os.path.join(BASE_PATH, 'ICPR02', 'kaggle')
EVAL_DIR    = os.path.join(BASE_PATH, 'ICPR02', 'kaggle', 'evaluation')

BAND_NAMES  = ['B1','B2','B3','B4','B5','B6','B7','B8','B8A','B9','B11','B12']
NUM_BANDS   = 12
IMG_SIZE    = 64     # Start small for stable training (increase later)
CLASS_NAMES = ['Aphid', 'Blast', 'RPH', 'Rust']
CLASS_MAP   = {n: i for i, n in enumerate(CLASS_NAMES)}
INV_MAP     = {i: n for n, i in CLASS_MAP.items()}


def build_labeled_data():
    """Build (file_paths, labels) from class folders."""
    paths, labels = [], []
    for cls in CLASS_NAMES:
        cls_dir = os.path.join(TRAIN_DIR, cls)
        if not os.path.exists(cls_dir):
            print(f"  ⚠️ Missing: {cls_dir}")
            continue
        for item in sorted(os.listdir(cls_dir)):
            paths.append(os.path.join(cls_dir, item))
            labels.append(CLASS_MAP[cls])
    labels = np.array(labels)
    print(f"📊 Labeled data: {len(paths)} samples")
    for cls in CLASS_NAMES:
        print(f"   {cls}: {(labels == CLASS_MAP[cls]).sum()}")
    return paths, labels


def build_test_data():
    """Build file paths for evaluation folder."""
    paths, ids = [], []
    if not os.path.exists(EVAL_DIR):
        print(f"⚠️ Eval dir missing: {EVAL_DIR}")
        return paths, ids
    for item in sorted(os.listdir(EVAL_DIR)):
        paths.append(os.path.join(EVAL_DIR, item))
        ids.append(item)
    print(f"📊 Test data: {len(paths)} samples")
    return paths, ids


class SentinelDataset(Dataset):
    """Loads Sentinel-2 images (folder-of-bands or single-file)."""
    def __init__(self, file_paths, labels=None, img_size=IMG_SIZE, augment=False):
        self.paths = file_paths
        self.labels = labels
        self.img_size = img_size
        self.augment = augment

    def _load(self, path):
        bands = []
        if os.path.isdir(path):
            for bn in BAND_NAMES:
                tif = os.path.join(path, f"{bn}.tif")
                if os.path.exists(tif):
                    with rasterio.open(tif) as src:
                        bands.append(src.read(1).astype(np.float32))
                else:
                    bands.append(None)
        else:
            fpath = path if os.path.isfile(path) else path + '.tif'
            if os.path.exists(fpath):
                with rasterio.open(fpath) as src:
                    data = src.read().astype(np.float32)
                    for i in range(data.shape[0]):
                        bands.append(data[i])
            while len(bands) < NUM_BANDS:
                bands.append(None)

        result = []
        for b in bands[:NUM_BANDS]:
            if b is not None:
                t = torch.from_numpy(b).unsqueeze(0).unsqueeze(0)
                t = F.interpolate(t, size=(self.img_size, self.img_size),
                                 mode='bilinear', align_corners=False)
                result.append(t.squeeze())
            else:
                result.append(torch.zeros(self.img_size, self.img_size))

        img = torch.stack(result, dim=0)  # (NUM_BANDS, H, W)

        # Normalize each band using percentile clipping (more robust than min-max)
        for i in range(img.shape[0]):
            b = img[i]
            valid = b[b > 0]
            if len(valid) > 0:
                lo = torch.quantile(valid, 0.02)
                hi = torch.quantile(valid, 0.98)
                if hi > lo:
                    img[i] = torch.clamp((b - lo) / (hi - lo), 0.0, 1.0)
                else:
                    img[i] = torch.zeros_like(b)
            else:
                img[i] = torch.zeros_like(b)
        return img

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = self._load(self.paths[idx])
        if self.augment:
            if torch.rand(1) > 0.5: img = torch.flip(img, [2])
            if torch.rand(1) > 0.5: img = torch.flip(img, [1])
            k = torch.randint(0, 4, (1,)).item()
            img = torch.rot90(img, k, [1, 2])
        if self.labels is not None:
            return img, self.labels[idx]
        return img


# --- SANITY CHECK: Load one sample and print stats ---
print("✅ Utilities ready!")
train_paths, train_labels = build_labeled_data()
test_paths, test_ids = build_test_data()

print("\n🔍 Sanity check — loading first sample:")
_ds = SentinelDataset(train_paths[:1])
_img = _ds[0]
print(f"   Shape: {_img.shape}")
print(f"   Dtype: {_img.dtype}")
for i, bn in enumerate(BAND_NAMES):
    b = _img[i]
    print(f"   {bn:5s}: min={b.min():.3f} max={b.max():.3f} mean={b.mean():.3f} nonzero={(b>0).sum().item()}")
del _ds, _img


In [ ]:
class CropDiseaseModel(nn.Module):
    """
    Pre-trained ImageNet backbone adapted for 12-channel satellite input.
    timm's in_chans parameter handles all architectures automatically.
    """
    def __init__(self, backbone='resnet50', num_classes=4, pretrained=True):
        super().__init__()
        self.model = timm.create_model(
            backbone,
            pretrained=pretrained,
            num_classes=num_classes,
            in_chans=NUM_BANDS
        )
        print(f"✅ {backbone}: {NUM_BANDS}-ch input, {num_classes} classes (pretrained={pretrained})")

    def forward(self, x):
        return self.model(x)


# Quick sanity test
_m = CropDiseaseModel('resnet50', pretrained=False)
_d = torch.randn(2, NUM_BANDS, IMG_SIZE, IMG_SIZE)
_o = _m(_d)
print(f"✅ Model test: {_d.shape} → {_o.shape}")
del _m, _d, _o

In [ ]:
BACKBONES = {
    'resnet50':        {'img_size': 64,  'bs': 32, 'lr': 3e-4},
    'efficientnet_b2': {'img_size': 64,  'bs': 32, 'lr': 3e-4},
    'convnext_tiny':   {'img_size': 64,  'bs': 16, 'lr': 1e-4},
    'swin_tiny_patch4_window7_224': {'img_size': 224, 'bs': 8, 'lr': 1e-4},
}

COMPARE_EPOCHS = 20
COMPARE_FOLDS  = 3

backbone_results = {}

for bb_name, bb_cfg in BACKBONES.items():
    print(f"\n{'🔷'*25}")
    print(f"🏗️ BACKBONE: {bb_name}")
    print(f"   img_size={bb_cfg['img_size']} | bs={bb_cfg['bs']} | lr={bb_cfg['lr']}")
    print(f"{'🔷'*25}")

    skf = StratifiedKFold(n_splits=COMPARE_FOLDS, shuffle=True, random_state=42)
    fold_accs, fold_f1s = [], []
    start_time = time.time()

    for fold, (tr_idx, va_idx) in enumerate(skf.split(train_paths, train_labels)):
        print(f"\n  📂 Fold {fold+1}/{COMPARE_FOLDS}")

        tr_paths_f = [train_paths[i] for i in tr_idx]
        va_paths_f = [train_paths[i] for i in va_idx]
        tr_labels_f = train_labels[tr_idx]
        va_labels_f = train_labels[va_idx]

        tr_loader = DataLoader(
            SentinelDataset(tr_paths_f, tr_labels_f, img_size=bb_cfg['img_size'], augment=True),
            batch_size=bb_cfg['bs'], shuffle=True, num_workers=2, pin_memory=True)
        va_loader = DataLoader(
            SentinelDataset(va_paths_f, va_labels_f, img_size=bb_cfg['img_size']),
            batch_size=bb_cfg['bs'], num_workers=2, pin_memory=True)

        try:
            model = CropDiseaseModel(bb_name, num_classes=4, pretrained=True).to(DEVICE)
        except Exception as e:
            print(f"  ❌ Failed: {e}")
            break

        # Class weights
        counts = np.bincount(tr_labels_f, minlength=4).astype(np.float32)
        weights = torch.tensor(counts.sum() / (4 * counts + 1e-6)).to(DEVICE)
        criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)

        # Single optimizer — same LR for all params (simpler, more stable)
        optimizer = torch.optim.AdamW(model.parameters(), lr=bb_cfg['lr'], weight_decay=1e-4)

        # Warmup + Cosine schedule
        warmup_epochs = 3
        def lr_lambda(epoch):
            if epoch < warmup_epochs:
                return (epoch + 1) / warmup_epochs
            return 0.5 * (1 + np.cos(np.pi * (epoch - warmup_epochs) / (COMPARE_EPOCHS - warmup_epochs)))
        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

        best_acc, best_f1 = 0, 0
        for ep in range(COMPARE_EPOCHS):
            model.train()
            ep_loss = 0
            for imgs, lbls in tr_loader:
                imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
                loss = criterion(model(imgs), lbls)
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                ep_loss += loss.item()
            scheduler.step()

            model.eval()
            vp, vt = [], []
            with torch.no_grad():
                for imgs, lbls in va_loader:
                    preds = model(imgs.to(DEVICE)).argmax(1).cpu().numpy()
                    vp.extend(preds); vt.extend(lbls.numpy())
            acc = accuracy_score(vt, vp)
            f1 = f1_score(vt, vp, average='macro')

            if acc > best_acc:
                best_acc = acc
                best_f1 = f1

            if (ep+1) % 5 == 0:
                lr_now = optimizer.param_groups[0]['lr']
                print(f"    Ep {ep+1:2d}/{COMPARE_EPOCHS} | Loss: {ep_loss/len(tr_loader):.4f} | "
                      f"Acc: {acc:.4f} | F1: {f1:.4f} | LR: {lr_now:.1e}")

        fold_accs.append(best_acc)
        fold_f1s.append(best_f1)
        print(f"  ✅ Fold {fold+1} → Acc: {best_acc:.4f} | F1: {best_f1:.4f}")
        del model; torch.cuda.empty_cache()

    elapsed = time.time() - start_time
    if fold_accs:
        backbone_results[bb_name] = {
            'mean_acc': np.mean(fold_accs),
            'std_acc': np.std(fold_accs),
            'mean_f1': np.mean(fold_f1s),
            'time_min': elapsed / 60,
            'cfg': bb_cfg,
        }

# --- Comparison Table ---
print(f"\n\n{'='*70}")
print(f"📊 BACKBONE COMPARISON ({COMPARE_FOLDS}-Fold, {COMPARE_EPOCHS} epochs)")
print(f"{'='*70}")
print(f"  {'Backbone':<35} {'Acc':>10} {'F1':>10} {'Time':>10}")
print("-" * 70)
best_bb, best_score = None, 0
for name, res in sorted(backbone_results.items(), key=lambda x: -x[1]['mean_f1']):
    print(f"  {name:<33} {res['mean_acc']:.4f}±{res['std_acc']:.3f}"
          f"  {res['mean_f1']:.4f}    {res['time_min']:.1f} min")
    if res['mean_f1'] > best_score:
        best_score = res['mean_f1']
        best_bb = name

print(f"\n🏆 BEST: {best_bb} (F1: {best_score:.4f})")

# Bar chart
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
names = list(backbone_results.keys())
accs = [backbone_results[n]['mean_acc'] for n in names]
f1s = [backbone_results[n]['mean_f1'] for n in names]
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']

for ax, vals, title in [(axes[0], accs, 'Accuracy'), (axes[1], f1s, 'F1 Score')]:
    bars = ax.bar(range(len(names)), vals, color=colors[:len(names)], edgecolor='black', lw=0.5)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=25, ha='right', fontsize=8)
    ax.set_ylabel(title); ax.set_title(f'{title} by Backbone', fontweight='bold')
    for i, v in enumerate(vals): ax.text(i, v+0.005, f'{v:.3f}', ha='center', fontsize=9)
plt.suptitle('🏗️ Backbone Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/backbone_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
BEST_BB   = best_bb
BEST_CFG  = backbone_results[BEST_BB]['cfg']
FT_EPOCHS = 40
N_FOLDS   = 5

print(f"🚀 Full training: {BEST_BB}")
print(f"   img_size={BEST_CFG['img_size']} | bs={BEST_CFG['bs']} | lr={BEST_CFG['lr']}")

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
fold_scores = []
all_val_preds, all_val_true = [], []

for fold, (tr_idx, va_idx) in enumerate(skf.split(train_paths, train_labels)):
    print(f"\n{'='*50}\n📂 FOLD {fold+1}/{N_FOLDS}\n{'='*50}")

    tr_paths_f = [train_paths[i] for i in tr_idx]
    va_paths_f = [train_paths[i] for i in va_idx]
    tr_labels_f = train_labels[tr_idx]
    va_labels_f = train_labels[va_idx]

    tr_loader = DataLoader(
        SentinelDataset(tr_paths_f, tr_labels_f, img_size=BEST_CFG['img_size'], augment=True),
        batch_size=BEST_CFG['bs'], shuffle=True, num_workers=2, pin_memory=True)
    va_loader = DataLoader(
        SentinelDataset(va_paths_f, va_labels_f, img_size=BEST_CFG['img_size']),
        batch_size=BEST_CFG['bs'], num_workers=2, pin_memory=True)

    model = CropDiseaseModel(BEST_BB, num_classes=4, pretrained=True).to(DEVICE)

    counts = np.bincount(tr_labels_f, minlength=4).astype(np.float32)
    weights = torch.tensor(counts.sum() / (4 * counts + 1e-6)).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)

    optimizer = torch.optim.AdamW(model.parameters(), lr=BEST_CFG['lr'], weight_decay=1e-4)
    warmup_epochs = 5
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        return 0.5 * (1 + np.cos(np.pi * (epoch - warmup_epochs) / (FT_EPOCHS - warmup_epochs)))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    best_acc = 0
    for ep in range(FT_EPOCHS):
        model.train()
        train_loss = 0
        for imgs, lbls in tr_loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            loss = criterion(model(imgs), lbls)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()
        scheduler.step()

        model.eval()
        vp, vt = [], []
        with torch.no_grad():
            for imgs, lbls in va_loader:
                preds = model(imgs.to(DEVICE)).argmax(1).cpu().numpy()
                vp.extend(preds); vt.extend(lbls.numpy())

        acc = accuracy_score(vt, vp)
        f1 = f1_score(vt, vp, average='macro')

        if acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(), f'/kaggle/working/best_fold{fold}.pth')

        if (ep+1) % 5 == 0 or ep == 0:
            print(f"  Ep {ep+1:2d}/{FT_EPOCHS} | Loss: {train_loss/len(tr_loader):.4f} | "
                  f"Acc: {acc:.4f} | F1: {f1:.4f} | Best: {best_acc:.4f}")

    fold_scores.append(best_acc)
    all_val_preds.extend(vp)
    all_val_true.extend(vt)
    print(f"\n  ✅ Fold {fold+1} Best: {best_acc:.4f}")
    print(classification_report(vt, vp, target_names=CLASS_NAMES))
    del model; torch.cuda.empty_cache()

print(f"\n{'='*50}\n📊 FINAL CV RESULTS ({BEST_BB})\n{'='*50}")
for i, s in enumerate(fold_scores):
    print(f"  Fold {i+1}: {s:.4f}")
print(f"  Mean:  {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")
print(f"  Overall F1: {f1_score(all_val_true, all_val_preds, average='macro'):.4f}")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(all_val_true, all_val_preds)
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title(f'Confusion Matrix — {BEST_BB}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print("🎯 GENERATING SUBMISSION")

if len(test_paths) > 0:
    test_ds = SentinelDataset(test_paths, img_size=BEST_CFG['img_size'])
    all_preds = []

    for fold in range(N_FOLDS):
        mp = f'/kaggle/working/best_fold{fold}.pth'
        if not os.path.exists(mp):
            continue

        model = CropDiseaseModel(BEST_BB, num_classes=4, pretrained=False).to(DEVICE)
        model.load_state_dict(torch.load(mp, map_location=DEVICE))
        model.eval()

        fold_preds = []
        with torch.no_grad():
            loader = DataLoader(test_ds, batch_size=BEST_CFG['bs'], num_workers=2)
            for imgs in loader:
                if isinstance(imgs, (list, tuple)):
                    imgs = imgs[0]
                pred = F.softmax(model(imgs.to(DEVICE)), dim=1).cpu().numpy()
                fold_preds.append(pred)

        all_preds.append(np.concatenate(fold_preds, axis=0))
        del model; torch.cuda.empty_cache()
        print(f"  ✅ Fold {fold+1} done")

    if all_preds:
        ensemble = np.mean(all_preds, axis=0)
        final_classes = ensemble.argmax(axis=1)

        submission = pd.DataFrame({
            'Id': test_ids,
            'Category': [INV_MAP[c] for c in final_classes]
        })
        submission.to_csv('/kaggle/working/submission.csv', index=False)

        print(f"\n✅ submission.csv saved! ({len(submission)} predictions)")
        print(f"\nDistribution:\n{submission['Category'].value_counts().to_string()}")
        print(f"\nFirst 10:\n{submission.head(10).to_string(index=False)}")

        max_conf = ensemble.max(axis=1)
        print(f"\n📊 Confidence: mean={max_conf.mean():.3f} | "
              f"min={max_conf.min():.3f} | max={max_conf.max():.3f}")
else:
    print("⚠️ No test data found.")

print("\n🎉 Done! Submit 'submission.csv' on the competition page.")